## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-16-21-54-01 +0000,chitri,Workers strike at 3 Chicago hotels,https://www.chicagotribune.com/2026/11/16/hote...
1,2026-11-16-21-51-00 +0000,wsj,U.S. Natural Gas Futures Settle Lower,https://www.wsj.com/finance/commodities-future...
2,2026-11-16-21-50-00 +0000,wsj,Oil Futures Retreat on Expected Global Surplus,https://www.wsj.com/finance/commodities-future...
3,2026-11-16-21-45-49 +0000,chitri,Pope Leo bids ‘hasta luego’ to his second home...,https://www.chicagotribune.com/2026/11/16/pope...
4,2026-11-16-21-44-09 +0000,bbc,Will enough Labour MPs back benefit cuts this ...,https://www.bbc.co.uk/news/videos/cvy0e9y18w5x...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2741/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
34,trump,35
138,nvidia,26
483,new,18
93,ai,16
139,china,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
129,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...,122
143,2026-11-16-17-02-28 +0000,chitri,President Trump backs campaign to reinstate Nv...,https://www.chicagotribune.com/2026/11/16/nvid...,115
118,2026-11-16-17-48-36 +0000,nyt,Trump Says He’s Open to Reinstatement of Jense...,https://www.nytimes.com/2026/11/16/us/politics...,103
197,2026-11-16-14-46-32 +0000,nypost,Trump ‘would not mind’ Jensen Huang reinstated...,https://nypost.com/2026/11/16/us-news/trump-wo...,102
278,2026-11-16-10-00-00 +0000,wapo,"Midterm losses, Nvidia scandal weaken Trump as...",https://www.washingtonpost.com/politics/2026/1...,96


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
129,122,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...
228,75,2026-11-16-12-33-00 +0000,wsj,"Nvidia Earnings, Anthropic IPO Ahead in New Te...",https://www.wsj.com/cfo-journal/nvidia-earning...
231,48,2026-11-16-12-04-00 +0000,wsj,A Texas-based U.S. attorney sent what he calle...,https://www.wsj.com/arts-culture/books/the-whi...
79,47,2026-11-16-19-30-30 +0000,bbc,Lindsay Clancy to face retrial next year for d...,https://www.bbc.co.uk/news/articles/cw5y1v8de3...
197,41,2026-11-16-14-46-32 +0000,nypost,Trump ‘would not mind’ Jensen Huang reinstated...,https://nypost.com/2026/11/16/us-news/trump-wo...
186,41,2026-11-16-14-58-12 +0000,bbc,Landmark Crimea bridge hit as Russia reports b...,https://www.bbc.co.uk/news/articles/ckx27v6dpj...
3,39,2026-11-16-21-45-49 +0000,chitri,Pope Leo bids ‘hasta luego’ to his second home...,https://www.chicagotribune.com/2026/11/16/pope...
115,37,2026-11-16-18-01-25 +0000,nypost,Passengers enduring 60-hour travel delays stil...,https://nypost.com/2026/11/16/business/passeng...
25,34,2026-11-16-21-21-03 +0000,nypost,JPMorgan Chase says in-house software issue im...,https://nypost.com/2026/11/16/business/jpmorga...
319,34,2026-11-16-02-11-18 +0000,nyt,"Miguel Ángel Félix Gallardo, Mexican Drug Lord...",https://www.nytimes.com/2026/11/15/world/ameri...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
